# Lab: Sentiment Analysis  
#  *******Data-Centric vs Model-Centric approaches




This lab gives an introduction to sentiment analysis approaches.

In this lab, we'll build a classifier for product reviews (restricted to the magazine category), like:

> Excellent! I look forward to every issue. I had no idea just how much I didn't know.  The letters from the subscribers are educational, too.

Label: ⭐️⭐️⭐️⭐️⭐️ (good)

> My son waited and waited, it took the 6 weeks to get delivered that they said it would but when it got here he was so dissapointed, it only took him a few minutes to read it.

Label: ⭐️ (bad)

We'll work with a dataset that has some issues, and we'll see how we can squeeze only so much performance out of the model by being clever about model choice, searching for better hyperparameters, etc. Then, we'll take a look at the data (as any good data scientist should), develop an understanding of the issues, and use simple approaches to improve the data. Finally, we'll see how improving the data can improve results.

---

**Course Code:** CS4082 Machine Learning

**Institution:** Effat University, Computer Science Department

**Student Name:** Ghala Alghamdi

**Student ID:** S23108212

**Instructor:** Dr. Naila Marir

**Date:** April 15, 2026

---

## Installing software

For this lab, you'll need to install [scikit-learn](https://scikit-learn.org/) and [pandas](https://pandas.pydata.org/). If you don't have them installed already, you can install them by running the following cell:

In [1]:
!pip install scikit-learn pandas

# Loading the data

First, let's load the train/test sets and take a look at the data.

In [2]:
import pandas as pd

In [3]:
train = pd.read_csv('reviews_train.csv')
test = pd.read_csv('reviews_test.csv')

test.sample(5)

,review,label
234,Love Golf Magazine,good
861,"The magazine is 90% ads, oh and even the artic...",bad
855,I look forward to receiving this magazine each...,bad
396,i love this magazine... it helps me from dress...,good
691,"I ordered 12/16/09, as if 02/12/10 (8 weeks) I...",bad


# Training a baseline model

There are many approaches for training a sequence classification model for text data. In this lab, we're giving you code that mirrors what you find if you look up [how to train a text classifier](https://scikit-learn.org/stable/tutorial/text_analytics/working_with_text_data.html), where we'll train an SVM on [tf-idf](https://en.wikipedia.org/wiki/Tf%E2%80%93idf) features (numeric representations of each text field based on word occurrences).

In [4]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.linear_model import SGDClassifier
from sklearn.pipeline import Pipeline

In [5]:
sgd_clf = Pipeline([
    ('vect', CountVectorizer()),
    ('tfidf', TfidfTransformer()),
    ('clf', SGDClassifier()),
])

In [6]:
_ = sgd_clf.fit(train['review'], train['label'])

## Evaluating model accuracy

In [7]:
from sklearn import metrics

In [8]:
def evaluate(clf):
    pred = clf.predict(test['review'])
    acc = metrics.accuracy_score(test['label'], pred)
    print(f'Accuracy: {100*acc:.1f}%')

In [9]:
evaluate(sgd_clf)

Accuracy: 76.9%


## Trying another model

76% accuracy is not great for this binary classification problem. Can you do better with a different model, or by tuning hyperparameters for the SVM trained with SGD?

# Exercise 1

Can you train a more accurate model on the dataset (without changing the dataset)? You might find this [scikit-learn classifier comparison](https://scikit-learn.org/stable/auto_examples/classification/plot_classifier_comparison.html) handy, as well as the [documentation for supervised learning in scikit-learn](https://scikit-learn.org/stable/supervised_learning.html).

One idea for a model you could try is a [naive Bayes classifier](https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.MultinomialNB.html).

You could also try experimenting with different values of the model hyperparameters, perhaps tuning them via a [grid search](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html).

Or you can even try training multiple different models and [ensembling their predictions](https://scikit-learn.org/stable/modules/ensemble.html#voting-classifier), a strategy often used to win prediction competitions like Kaggle.

**Advanced:** If you want to be more ambitious, you could try an even fancier model, like training a Transformer neural network. If you go with that, you'll want to fine-tune a pre-trained model. This [guide from HuggingFace](https://huggingface.co/docs/transformers/training) may be helpful.

In [10]:
import pandas as pd
from sklearn.base import clone
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def compare_models(models, X_train, y_train, X_test, y_test):
    """
    Train and evaluate multiple classification models.

    Parameters
    ----------
    models : dict or list
        Dictionary like {"Model Name": model}
        or list like [model1, model2, ...]
    X_train : training features
    y_train : training labels
    X_test : testing features
    y_test : testing labels

    Returns
    -------
    pd.DataFrame
        Table containing Model, Accuracy, Precision, Recall, F1-score
    """

    results = []

    #if models are given as a list, convert to a dictionary
    if isinstance(models, list):
        models = {model.__class__.__name__: model for model in models}

    for model_name, model in models.items():
        clf = clone(model)   #avoids modifying the original model
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)

        results.append({
            "Model name": model_name,
            "Accuracy": accuracy_score(y_test, y_pred),
            "Precision": precision_score(y_test, y_pred, average="binary", pos_label="good"),
            "Recall": recall_score(y_test, y_pred, average="binary", pos_label="good"),
            "F1-score": f1_score(y_test, y_pred, average="binary", pos_label="good")
        })

    results_df = pd.DataFrame(results)
    return results_df.sort_values(by="F1-score", ascending=False).reset_index(drop=True)

In [11]:
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier, LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC

models = {
    "Logistic Regression": make_pipeline(
        TfidfVectorizer(stop_words="english", ngram_range=(1, 2), min_df=2),
        LogisticRegression(max_iter=3000, random_state=42)
    ),

    "Linear SVC": make_pipeline(
        TfidfVectorizer(stop_words="english", ngram_range=(1, 2), min_df=2),
        LinearSVC()
    ),

    "Multinomial Naive Bayes": make_pipeline(
        TfidfVectorizer(stop_words="english", ngram_range=(1, 2), min_df=2),
        MultinomialNB()
    ),

    "SGD Classifier": make_pipeline(
        TfidfVectorizer(stop_words="english", ngram_range=(1, 2), min_df=2),
        SGDClassifier(random_state=42)
    )
}

comparison_df = compare_models(
    models,
    train["review"], train["label"],
    test["review"], test["label"]
)

comparison_df

,Model name,Accuracy,Precision,Recall,F1-score
0,Multinomial Naive Bayes,0.832,0.830677,0.834,0.832335
1,Logistic Regression,0.822,0.834025,0.804,0.818737
2,SGD Classifier,0.771,0.782881,0.750,0.766088
3,Linear SVC,0.728,0.730769,0.722,0.726358


## Taking a closer look at the training data

Let's actually take a look at some of the training data:

In [12]:
train.head()

,review,label
0,Based on all the negative comments about Taste...,good
1,I still have not received this. Obviously I c...,bad
2,</tr>The magazine is not worth the cost of sub...,good
3,This magazine is basically ads. Kindve worthle...,bad
4,"The only thing I've recieved, so far, is the b...",bad


Zooming in on one particular data point:

In [13]:
print(train.iloc[0].to_dict())

{'review': "Based on all the negative comments about Taste of Home, I will not subscribeto the magazine. In the past it was a great read.\nSorry it, too, has gone the 'way of the wind'.<br>o-p28pass4 </br>", 'label': 'good'}


This data point is labeled "good", but it's clearly a negative review. Also, it looks like there's some funny HTML stuff at the end.

# Exercise 2

Take a look at some more examples in the dataset. Do you notice any patterns with bad data points?

In [14]:
#look at some examples that may contain HTML-like text
html_examples = train[train["review"].str.contains(r"<[^>]+>", regex=True, na=False)]
html_examples.head(10)

,review,label
0,Based on all the negative comments about Taste...,good
2,</tr>The magazine is not worth the cost of sub...,good
5,"The magazines are great, but I never received ...",good
10,"</div>It's not the fault of the magazine, I ju...",good
11,<li>dispatchEventBest magazine for current and...,bad
12,<li>onEmptiedBoth my husband and I really enjo...,bad
13,This magazine is filled with amazing recipes. ...,bad
17,</div>purchased for my wife who loves reading ...,bad
19,"<h4 class=""signature"">36 page pamphlet with LB...",good
22,No iPhone supportorg.python.core</FONT>,good


> **Pattern noticed:**
Many bad data points contain HTML tags or broken scraped text.
These examples often have incorrect or nonsense labels.
So the issue seems to be dirty data caused by web-scraping/HTML content,
not only a weak model.

## Issues in the data

It looks like there's some funny HTML tags in our dataset, and those datapoints have nonsense labels. Maybe this dataset was collected by scraping the internet, and the HTML wasn't quite parsed correctly in all cases.

# Exercise 3

To address this, a simple approach we might try is to throw out the bad data points, and train our model on only the "clean" data.

Come up with a simple heuristic to identify data points containing HTML, and filter out the bad data points to create a cleaned training set.

In [15]:
import re

def is_bad_data(review: str) -> bool:
    if pd.isna(review):
        return True

    #mark as bad if it contains HTML tags
    return bool(re.search(r"<[^>]+>", review))

## Creating the cleaned training set

In [16]:
train_clean = train[~train['review'].map(is_bad_data)]

## Evaluating a model trained on the clean training set

In [17]:
from sklearn import clone

In [18]:
sgd_clf_clean = clone(sgd_clf)

In [19]:
_ = sgd_clf_clean.fit(train_clean['review'], train_clean['label'])

This model should do significantly better:

In [20]:
evaluate(sgd_clf_clean)

Accuracy: 97.0%
